# ✈️ AeroLex — Building a Production-Grade RAG Pipeline

A hands-on build of a multi-stage Retrieval-Augmented Generation system over
**airline passenger-rights law** (EU 261/2004, US DOT 14 CFR 260, Delta Contracts
of Carriage).

**The pipeline:**
```
question
  └─▶ condense follow-up            (conversational memory)       [#8]
  └─▶ BM25 lexical  +  Gemini dense  (hybrid retrieval)           [#2,#3,#4]
  └─▶ reciprocal rank fusion         (merge the two)              [#5]
  └─▶ FLAIR feedback re-scoring      (learn from 👍/👎)            [#10]
  └─▶ cross-encoder re-ranking       (precise top-k)              [#6]
  └─▶ grounded answer + [[n]] cites  (+ long-term profile)        [#7,#9]
```

**How to run:** pick the **`Python (aerolex RAG)`** kernel, then run top to bottom.
Everything works offline (extractive answers); with a `GEMINI_API_KEY` you get full
generation. You built each stage? The final section assembles them with the
`aerolex` library.


## 0 · Setup

In [ ]:
import numpy as np
from aerolex import config
from aerolex.loaders import load_documents
from aerolex.chunking import chunk_corpus, load_chunks
from aerolex.store import InMemoryVectorStore

key = config.load_api_key()
print("GEMINI_API_KEY:", "✓ found" if key else "✗ not found — offline fallback mode")
print("gen model:", config.GEN_MODEL, "| embed model:", config.EMBED_MODEL, "| dim:", config.EMBED_DIM)

## 1 · Hierarchical Chunking  ·  Feature #1

Legal text is a **hierarchy** (Article / § / RULE → paragraph). We split on those
*logical* boundaries — never fixed-size windows — and attach citation metadata, so
an answer can cite *"EU 261/2004 — Article 7"* instead of *"chunk 42"*.

In [ ]:
docs = load_documents()
for d in docs:
    print(f"{d.doc_id:16s} {d.doc_type:6s} {len(d.text):>7,d} chars  {d.title}")

# Prefer the pre-built cache (instant); otherwise chunk live.
try:
    chunks = load_chunks()
    print(f"\nloaded {len(chunks)} chunks from cache")
except FileNotFoundError:
    chunks = chunk_corpus(docs)
    print(f"\nchunked live: {len(chunks)} chunks")

by_id = {c.chunk_id: c for c in chunks}
art7 = next(c for c in chunks if c.section_label == "Article 7")
print("\nExample chunk —", art7.citation)
print(art7.text[:320], "…")

## 2 · Lexical Search with BM25  ·  Feature #2

Dense search blurs exact tokens, but statute codes must match **exactly**
(`§ 260.6`, `261/2004`). BM25 is a sparse bag-of-words scorer — the trick is a
**tokenizer that preserves those references** as single tokens.

### 🛠️ TODO: write `tokenize`

In [ ]:
import re

# TODO: build a regex so these survive as single tokens:
#   words (letters), the section sign §  (\u00A7), and numbers like 260.6 / 261/2004 / 14
# Hint: one pattern with alternation ( a | b | c ), applied to text.lower().
def tokenize(text):
    raise NotImplementedError("your turn — make the assert below pass")

assert tokenize("See § 260.6 and 261/2004") == ["see", "\u00A7", "260.6", "and", "261/2004"]
print("tokenize OK")

In [ ]:
from aerolex.bm25 import BM25Index

bm25 = BM25Index([c.chunk_id for c in chunks], [c.searchable_text for c in chunks])
for cid, score in bm25.search("§ 260.6 refund for a cancelled flight", top_k=3):
    print(f"{score:6.2f}  {by_id[cid].citation}")

## 3 · Dense Semantic Search + In-Memory Store  ·  Features #3 & #4

Gemini embeds every chunk into a vector; the whole corpus is one NumPy matrix.
Similarity is just a **normalized dot product** (cosine) — no vector database.

We load **pre-computed** embeddings from the cache so we don't embed 200+ chunks
live (the free tier is rate-limited).

In [ ]:
store = InMemoryVectorStore.load()          # cache/embeddings.npz
print("vector store:", len(store), "vectors ×", store.matrix.shape[1], "dims")

### 🛠️ TODO: write `cosine_search` (normalized dot product top-k)

In [ ]:
def cosine_search(query_vec, matrix, ids, top_k=5):
    # TODO: 1) L2-normalize query_vec and each row of matrix
    #       2) scores = matrix @ query   (cosine, since normalized)
    #       3) return the top_k (id, score) pairs, highest first
    raise NotImplementedError("your turn!")

_hit = cosine_search(store.matrix[0], store.matrix, store.ids, top_k=1)[0]
assert _hit[0] == store.ids[0] and abs(_hit[1] - 1.0) < 1e-4
print("cosine_search OK")

In [ ]:
# Embed a real query (1 API call) and search. Needs a key; else we reuse a chunk vector.
query = "How much compensation for a long delayed flight under EU261?"
if key:
    from aerolex.embeddings import GeminiEmbedder
    qvec = GeminiEmbedder().embed_query(query)
else:
    print("(no key — demoing with an existing chunk vector as the 'query')")
    qvec = store.matrix[store.ids.index(art7.chunk_id)]

dense_hits = cosine_search(qvec, store.matrix, store.ids, top_k=5)
for cid, s in dense_hits:
    print(f"{s:5.3f}  {by_id[cid].citation}")

## 4 · Reciprocal Rank Fusion  ·  Feature #5

BM25 scores and cosine scores live on different scales, so we fuse by **rank**:
each list contributes `1 / (k + rank)` to an id's score (`k = 60`). Documents that
rank well in *either* retriever bubble up.

### 🛠️ TODO: write `rrf`

In [ ]:
def rrf(rankings, k=60, top_k=10):
    # TODO: for each ranked list of ids, add 1/(k + rank + 1) to that id's score
    #       (rank starts at 0). Return the top_k (id, score) pairs, highest first.
    raise NotImplementedError("your turn!")

assert rrf([["b", "a", "c"], ["b", "a", "d"]])[0][0] == "b"
print("rrf OK")

In [ ]:
bm25_ids = [cid for cid, _ in bm25.search(query, top_k=20)]
dense_ids = [cid for cid, _ in dense_hits]
fused = rrf([bm25_ids, dense_ids], k=config.RRF_K, top_k=10)
print("Fused top-5:")
for cid, s in fused[:5]:
    print(f"{s:.4f}  {by_id[cid].citation}")

## 5 · Cross-Encoder Re-Ranking  ·  Feature #6

First-stage retrieval scores query and document *independently*. A cross-encoder
reads them **together** (deep cross-attention) and is far more precise — so we run
it only over the small fused pool. (Model is pre-cached; runs locally.)

In [ ]:
from aerolex.rerank import CrossEncoderReranker

candidates = [(cid, by_id[cid].searchable_text) for cid, _ in fused]
reranked = CrossEncoderReranker().rerank(query, candidates, top_k=config.RERANK_TOP_K)
print("Re-ranked top chunks:")
for cid, s in reranked:
    print(f"{s:7.2f}  {by_id[cid].citation}")
final_chunks = [by_id[cid] for cid, _ in reranked]

## 6 · Grounded Answer with Inline Citations  ·  Feature #7

The model answers **only** from numbered passages and tags each claim with `[[n]]`.
A regex post-processor then validates those tags against the real passages and
**drops hallucinated ones** — the difference between "looks cited" and "is cited".

### 🛠️ TODO: write `resolve_citations`

In [ ]:
CITE = re.compile(r"\[\[(\d+)\]\]")

def resolve_citations(answer, index):
    # TODO: keep only [[n]] whose n is a key in `index`; delete the rest.
    # Return (cleaned_answer, ["[[n]] <citation>", ...] for the n's actually used).
    raise NotImplementedError("your turn!")

_c, _r = resolve_citations("A [[1]] and B [[9]].", {1: "EU261 Art 7", 2: "EU261 Art 5"})
assert "[[9]]" not in _c and _r == ["[[1]] EU261 Art 7"]
print("resolve_citations OK")

In [ ]:
from aerolex.generate import Generator

result = Generator().generate(query, final_chunks)
print("OFFLINE" if result["offline"] else "GEMINI", "answer:\n")
print(result["answer"])
print("\nReferences:")
for r in result["references"]:
    print("  ", r)

## 7 · Conversational Memory  ·  Feature #8

Follow-ups like *"what about in the US?"* are meaningless to a retriever alone. We
**condense** the chat history + follow-up into a standalone query before searching.

In [ ]:
from aerolex.memory import ConversationMemory

mem = ConversationMemory()
mem.add_user(query)
mem.add_assistant(result["answer"][:160])
print("follow-up : what about in the US?")
print("standalone:", mem.condense("what about in the US?"))

## 8 · Long-Term Persistence  ·  Feature #9

Facts a user states persist across sessions in SQLite and can be injected into
future prompts.

In [ ]:
from aerolex.persistence import ProfileStore

prof = ProfileStore()
prof.remember("demo_user", "home_airport", "JFK")
prof.remember("demo_user", "status", "Delta Platinum Medallion")
print(prof.context_block("demo_user"))

## 9 · Adaptive Feedback — FLAIR  ·  Feature #10

We log `(query, chunk, 👍/👎)` triplets and nudge future rankings by **query-weighted
votes**: a past vote counts only when the new query is similar to the one it was
cast for — so feedback generalizes to unseen-but-similar questions.

In [ ]:
from aerolex.feedback import FeedbackStore

fb = FeedbackStore()
top_id = reranked[0][0]
fb.record(query, qvec, top_id, +1)                 # 👍 the best chunk
ranked = [(cid, s) for cid, s in reranked]
adjusted = fb.apply(ranked, qvec)
print("before:", [by_id[c].section_label for c, _ in ranked])
print("after :", [by_id[c].section_label for c, _ in adjusted])

## 10 · The Full Pipeline

Now assemble every stage with the `aerolex` library — the productionized version of
what you just built — and ask a question end to end.

In [ ]:
from aerolex.pipeline import RagPipeline

if key:
    from aerolex.embeddings import GeminiEmbedder
    pipe = RagPipeline(
        chunks=chunks,
        store=store,
        bm25=bm25,
        embedder=GeminiEmbedder(),
        reranker=CrossEncoderReranker(),
        generator=Generator(),
    )
    out = pipe.answer("If my 4000 km flight is cancelled, what am I owed?",
                      memory=mem, profile=prof, user_id="demo_user")
    print(out["answer"])
    print("\nReferences:")
    for r in out["references"]:
        print("  ", r)
else:
    print("Set GEMINI_API_KEY to run the full generative pipeline.")

## 11 · Where to go next — citations, state of the art

What we built is **Level 1** citation: the model self-tags `[[n]]`, and we validate
the *number is real*. State-of-the-art adds **verification**:

- **Level 2** structured output — model returns `{sentence, support_ids, quote}`.
- **Level 3** quote-grounded spans — match the quote back into the source for exact
  offsets; drop unverifiable quotes.
- **Level 4** NLI / entailment (ALCE) — check the passage actually *entails* the
  claim; score citation **precision & recall**.

The retrieval stack (chunking → hybrid → fusion → re-rank) is where most answer
quality is won. 🎓 You built all of it.